In [1]:
# ============================================================
# SI7-A — MRB1-Informed LinTS Context Builder
# ============================================================
#
# PURPOSE
# -------
# Replace weak/ad-hoc LinTS context features with a compact,
# causally valid context built from:
#
#   1. Frozen MD6 selector probabilities
#   2. Current mastery_before
#   3. PREVIOUS attempt evaluator score
#   4. PREVIOUS attempt mastery change
#   5. PREVIOUS tutor-response pedagogical quality from MRB1:
#        - Mistake Identification
#        - Mistake Location
#        - Providing Guidance
#        - Actionability
#   6. has_previous_feedback flag
#
# IMPORTANT TEMPORAL RULE
# -----------------------
# Current-attempt evaluator score and current mastery_after do
# NOT exist when LinTS makes the current decision.
#
# Therefore they are NOT current context features.
#
# Current attempt:
#     mastery_before_k
#     + previous-attempt feedback
#     + current MD6 probabilities
#           ↓
#         LinTS
#           ↓
#     current policy/action
#           ↓
#     tutor + student interaction
#           ↓
#     evaluator_score_k + mastery_after_k + MRB1 scores_k
#
# Those newly observed values can affect the NEXT attempt.
#
# PRIMARY REWARD CONTRACT
# -----------------------
#     reward_k = evaluator_score_k / 3
#
# mastery_delta_k is preserved as a secondary learning outcome
# and next-attempt context signal. No arbitrary 0.8/0.2 reward
# mixing is used in the main design.
#
# This notebook does not train LinTS.
# It freezes and validates the context/reward contract first.
# ============================================================

import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Optional

import numpy as np

RESULT_DIR = Path(
    "/kaggle/working/si7a_context"
)
RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MOVE_ORDER = [
    "generic",
    "probing",
    "focus",
    "telling",
]

CRITIC_ORDER = [
    "mistake_identification",
    "mistake_location",
    "providing_guidance",
    "actionability",
]

FEATURE_NAMES = [
    # Current frozen selector state.
    "md6_p_generic",
    "md6_p_probing",
    "md6_p_focus",
    "md6_p_telling",

    # Current student state.
    "mastery_before",

    # Previous-attempt outcome/context only.
    "prev_evaluator_rate",
    "prev_mastery_delta",

    # Previous tutor-response pedagogical execution.
    "prev_mistake_identification",
    "prev_mistake_location",
    "prev_providing_guidance",
    "prev_actionability",

    # Distinguishes real zeros from "no prior attempt yet".
    "has_previous_feedback",
]

print("Feature dimension:", len(FEATURE_NAMES))
print("Feature names:")
for i, name in enumerate(FEATURE_NAMES):
    print(f"  {i:2d}: {name}")


Feature dimension: 12
Feature names:
   0: md6_p_generic
   1: md6_p_probing
   2: md6_p_focus
   3: md6_p_telling
   4: mastery_before
   5: prev_evaluator_rate
   6: prev_mastery_delta
   7: prev_mistake_identification
   8: prev_mistake_location
   9: prev_providing_guidance
  10: prev_actionability
  11: has_previous_feedback


In [2]:
# ============================================================
# DATA CONTRACTS
# ============================================================

@dataclass
class PreviousAttemptFeedback:
    evaluator_score: int
    mastery_before: float
    mastery_after: float

    # MRB1 expected scores in [0, 1].
    mistake_identification: float
    mistake_location: float
    providing_guidance: float
    actionability: float


def _check_unit_interval(
    name,
    value,
):
    value = float(value)

    if not (0.0 <= value <= 1.0):
        raise ValueError(
            f"{name} must be in [0,1], got {value}"
        )

    return value


def validate_md6_probabilities(
    probs: Dict[str, float]
):
    if set(probs.keys()) != set(MOVE_ORDER):
        raise ValueError(
            "MD6 probabilities must contain exactly "
            f"{MOVE_ORDER}; got {sorted(probs.keys())}"
        )

    ordered = np.asarray(
        [float(probs[m]) for m in MOVE_ORDER],
        dtype=np.float64,
    )

    if np.any(ordered < 0.0):
        raise ValueError(
            "MD6 probabilities cannot be negative."
        )

    if not np.isclose(
        ordered.sum(),
        1.0,
        atol=1e-5,
    ):
        raise ValueError(
            "MD6 probabilities must sum to 1. "
            f"Got {ordered.sum()}"
        )

    return ordered


In [3]:
# ============================================================
# MRB1 PROBABILITIES -> CONTINUOUS PEDAGOGICAL SCORE
# ============================================================
#
# MRB1-DIAG validated the expected-score ordering:
#
#   0.0 * P(No)
# + 0.5 * P(To some extent)
# + 1.0 * P(Yes)
#
# This function is the only transformation LinTS needs from the
# critic's 3-class probability output.
# ============================================================

MRB1_LABELS = [
    "No",
    "To some extent",
    "Yes",
]


def mrb1_expected_score(
    probabilities: Dict[str, float]
):
    if set(probabilities.keys()) != set(MRB1_LABELS):
        raise ValueError(
            "MRB1 probabilities must contain exactly "
            f"{MRB1_LABELS}"
        )

    p_no = float(
        probabilities["No"]
    )
    p_partial = float(
        probabilities["To some extent"]
    )
    p_yes = float(
        probabilities["Yes"]
    )

    if min(
        p_no,
        p_partial,
        p_yes,
    ) < 0.0:
        raise ValueError(
            "MRB1 probabilities cannot be negative."
        )

    total = (
        p_no
        + p_partial
        + p_yes
    )

    if not np.isclose(
        total,
        1.0,
        atol=1e-5,
    ):
        raise ValueError(
            "MRB1 probabilities must sum to 1. "
            f"Got {total}"
        )

    return float(
        0.5 * p_partial
        + p_yes
    )


# Sanity checks.
assert np.isclose(
    mrb1_expected_score({
        "No": 1.0,
        "To some extent": 0.0,
        "Yes": 0.0,
    }),
    0.0,
)

assert np.isclose(
    mrb1_expected_score({
        "No": 0.0,
        "To some extent": 1.0,
        "Yes": 0.0,
    }),
    0.5,
)

assert np.isclose(
    mrb1_expected_score({
        "No": 0.0,
        "To some extent": 0.0,
        "Yes": 1.0,
    }),
    1.0,
)

print("MRB1 expected-score transformation: PASSED")


MRB1 expected-score transformation: PASSED


In [4]:
# ============================================================
# FINAL CONTEXT BUILDER
# ============================================================

def build_lints_context(
    md6_probabilities: Dict[str, float],
    mastery_before: float,
    previous_feedback: Optional[
        PreviousAttemptFeedback
    ] = None,
):
    """
    Returns a float64 NumPy context vector in FEATURE_NAMES order.

    Causality:
    - md6_probabilities: current decision
    - mastery_before: current decision
    - previous_feedback: previous attempt only
    - NO current evaluator/mastery_after is accepted here
    """

    md6 = validate_md6_probabilities(
        md6_probabilities
    )

    mastery_before = _check_unit_interval(
        "mastery_before",
        mastery_before,
    )

    if previous_feedback is None:
        previous = np.zeros(
            7,
            dtype=np.float64,
        )

        # Seven slots:
        # eval, mastery_delta, MI, ML, PG, AC, flag
        previous[-1] = 0.0

    else:
        if previous_feedback.evaluator_score not in {
            0, 1, 2, 3
        }:
            raise ValueError(
                "evaluator_score must be one of {0,1,2,3}"
            )

        prev_mastery_before = _check_unit_interval(
            "previous.mastery_before",
            previous_feedback.mastery_before,
        )

        prev_mastery_after = _check_unit_interval(
            "previous.mastery_after",
            previous_feedback.mastery_after,
        )

        mastery_delta = (
            prev_mastery_after
            - prev_mastery_before
        )

        # delta is naturally in [-1,1].
        mastery_delta = float(
            np.clip(
                mastery_delta,
                -1.0,
                1.0,
            )
        )

        mi = _check_unit_interval(
            "previous.mistake_identification",
            previous_feedback.mistake_identification,
        )

        ml = _check_unit_interval(
            "previous.mistake_location",
            previous_feedback.mistake_location,
        )

        pg = _check_unit_interval(
            "previous.providing_guidance",
            previous_feedback.providing_guidance,
        )

        ac = _check_unit_interval(
            "previous.actionability",
            previous_feedback.actionability,
        )

        previous = np.asarray(
            [
                previous_feedback.evaluator_score / 3.0,
                mastery_delta,
                mi,
                ml,
                pg,
                ac,
                1.0,
            ],
            dtype=np.float64,
        )

    context = np.concatenate(
        [
            md6,
            np.asarray(
                [mastery_before],
                dtype=np.float64,
            ),
            previous,
        ]
    )

    assert context.shape == (
        len(FEATURE_NAMES),
    )

    assert np.isfinite(
        context
    ).all()

    return context


In [5]:
# ============================================================
# PRIMARY REWARD CONTRACT
# ============================================================
#
# Main experiment:
#
#   reward = evaluator_score / 3
#
# The reward is intentionally independent of MRB1.
# MRB1 describes pedagogical execution; evaluator score measures
# student learning outcome.
#
# mastery delta remains:
#   - logged as a secondary outcome
#   - available to the NEXT attempt's context
#   - available for a future reward ablation
# ============================================================

def primary_reward(
    evaluator_score: int
):
    if evaluator_score not in {
        0, 1, 2, 3
    }:
        raise ValueError(
            "evaluator_score must be in {0,1,2,3}"
        )

    return float(
        evaluator_score / 3.0
    )


for score, expected in [
    (0, 0.0),
    (1, 1/3),
    (2, 2/3),
    (3, 1.0),
]:
    assert np.isclose(
        primary_reward(score),
        expected,
    )

print("Primary reward contract: PASSED")


Primary reward contract: PASSED


In [6]:
# ============================================================
# ATTEMPT 0 — NO PREVIOUS FEEDBACK
# ============================================================

md6_attempt0 = {
    "generic": 0.12,
    "probing": 0.25,
    "focus": 0.51,
    "telling": 0.12,
}

x0 = build_lints_context(
    md6_probabilities=md6_attempt0,
    mastery_before=0.42,
    previous_feedback=None,
)

print("Attempt-0 context:")
for name, value in zip(
    FEATURE_NAMES,
    x0,
):
    print(
        f"{name:32s} {value:.4f}"
    )

assert x0[
    FEATURE_NAMES.index(
        "has_previous_feedback"
    )
] == 0.0

# All previous-feedback slots must be zero.
for feature in [
    "prev_evaluator_rate",
    "prev_mastery_delta",
    "prev_mistake_identification",
    "prev_mistake_location",
    "prev_providing_guidance",
    "prev_actionability",
]:
    assert x0[
        FEATURE_NAMES.index(feature)
    ] == 0.0

print("\nAttempt-0 cold-start context: PASSED")


Attempt-0 context:
md6_p_generic                    0.1200
md6_p_probing                    0.2500
md6_p_focus                      0.5100
md6_p_telling                    0.1200
mastery_before                   0.4200
prev_evaluator_rate              0.0000
prev_mastery_delta               0.0000
prev_mistake_identification      0.0000
prev_mistake_location            0.0000
prev_providing_guidance          0.0000
prev_actionability               0.0000
has_previous_feedback            0.0000

Attempt-0 cold-start context: PASSED


In [7]:
# ============================================================
# ATTEMPT 1 — PREVIOUS FEEDBACK NOW EXISTS
# ============================================================
#
# Example:
# Attempt 0 ended with:
#   evaluator = 2/3
#   mastery 0.42 -> 0.55
#   MRB1 scored the tutor response
#
# Attempt 1 may use those values because they are now historical.
# ============================================================

previous0 = PreviousAttemptFeedback(
    evaluator_score=2,
    mastery_before=0.42,
    mastery_after=0.55,

    mistake_identification=0.84,
    mistake_location=0.73,
    providing_guidance=0.79,
    actionability=0.68,
)

md6_attempt1 = {
    "generic": 0.10,
    "probing": 0.31,
    "focus": 0.47,
    "telling": 0.12,
}

x1 = build_lints_context(
    md6_probabilities=md6_attempt1,
    mastery_before=0.55,
    previous_feedback=previous0,
)

print("Attempt-1 context:")
for name, value in zip(
    FEATURE_NAMES,
    x1,
):
    print(
        f"{name:32s} {value:.4f}"
    )

assert np.isclose(
    x1[
        FEATURE_NAMES.index(
            "prev_evaluator_rate"
        )
    ],
    2/3,
)

assert np.isclose(
    x1[
        FEATURE_NAMES.index(
            "prev_mastery_delta"
        )
    ],
    0.13,
)

assert x1[
    FEATURE_NAMES.index(
        "has_previous_feedback"
    )
] == 1.0

print("\nAttempt-1 temporal context: PASSED")


Attempt-1 context:
md6_p_generic                    0.1000
md6_p_probing                    0.3100
md6_p_focus                      0.4700
md6_p_telling                    0.1200
mastery_before                   0.5500
prev_evaluator_rate              0.6667
prev_mastery_delta               0.1300
prev_mistake_identification      0.8400
prev_mistake_location            0.7300
prev_providing_guidance          0.7900
prev_actionability               0.6800
has_previous_feedback            1.0000

Attempt-1 temporal context: PASSED


In [8]:
# ============================================================
# CAUSALITY GUARDS
# ============================================================
#
# Explicitly verify the builder signature does not accept:
# - current mastery_after
# - current evaluator_score
# - future student response
#
# This protects the online decision from outcome leakage.
# ============================================================

import inspect

signature = inspect.signature(
    build_lints_context
)

accepted_parameters = set(
    signature.parameters.keys()
)

forbidden_current_inputs = {
    "mastery_after",
    "evaluator_score",
    "current_evaluator_score",
    "current_mastery_after",
    "future_student_response",
}

assert accepted_parameters.isdisjoint(
    forbidden_current_inputs
)

print(
    "Accepted context-builder parameters:",
    sorted(accepted_parameters),
)
print("Current-outcome leakage guard: PASSED")


Accepted context-builder parameters: ['mastery_before', 'md6_probabilities', 'previous_feedback']
Current-outcome leakage guard: PASSED


In [9]:
# ============================================================
# SAVE FROZEN CONTEXT SPECIFICATION
# ============================================================

spec = {
    "stage": "SI7-A",
    "name": "MRB1-informed LinTS context",
    "feature_dimension": len(FEATURE_NAMES),
    "feature_names": FEATURE_NAMES,

    "current_selection_inputs": {
        "md6_probabilities": MOVE_ORDER,
        "mastery_before": "[0,1]",
    },

    "previous_attempt_inputs": {
        "evaluator_score": "{0,1,2,3} -> divided by 3",
        "mastery_delta": "previous mastery_after - previous mastery_before",
        "mrb1_scores": CRITIC_ORDER,
        "has_previous_feedback": "{0,1}",
    },

    "mrb1_score_formula": (
        "0.0*P(No) + "
        "0.5*P(To some extent) + "
        "1.0*P(Yes)"
    ),

    "primary_reward": (
        "current evaluator_score / 3"
    ),

    "secondary_outcome": (
        "current mastery_after - current mastery_before"
    ),

    "forbidden_current_context": [
        "current evaluator score",
        "current mastery_after",
        "future student response",
        "future MRB1 score",
    ],

    "notes": [
        "MRB1 critic is frozen before LinTS learning.",
        "MD6 selector is frozen before LinTS learning.",
        "Current attempt outcomes enter only after the decision.",
        "They may influence the next attempt after becoming historical.",
        "No hand-built question/length heuristic is required.",
    ],
}

spec_path = (
    RESULT_DIR
    / "si7a_mrb1_lints_context_spec.json"
)

with spec_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        spec,
        f,
        indent=2,
    )

print("Saved:")
print(spec_path)

print("\n" + "=" * 78)
print("SI7-A VALIDATION PASSED")
print("=" * 78)
print("Context dimension:", len(FEATURE_NAMES))
print("Primary reward: evaluator_score / 3")
print("Current-outcome leakage: BLOCKED")


Saved:
/kaggle/working/si7a_context/si7a_mrb1_lints_context_spec.json

SI7-A VALIDATION PASSED
Context dimension: 12
Primary reward: evaluator_score / 3
Current-outcome leakage: BLOCKED
